### 🔴 Section 1: Implement for MA's

#### ⚪ 1. Libraries

In [ ]:
import pandas as pd
import scipy.io
import numpy as np
from scipy.io import loadmat
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial.distance import cdist
from sklearn.metrics import roc_auc_score, average_precision_score

#### ⚪ 2. Extract indices from .csv file

In [3]:
# --- Configuration ---
csv_file_path = '/media/mine/Mine/3.Main-Ubuntu-Envs/2.Thesis-Env/3.Datasets/mammographyori.csv'

def get_ma_indices(df, feature_name, low, high):
    condition = (df[feature_name] >= low) & (df[feature_name] <= high)
    subset = df[condition]
    indices = subset.index.to_numpy()
    counts = subset['label'].value_counts()
    return indices, counts

df_original = pd.read_csv(csv_file_path, header=None)
df_original.columns = ['c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'label']
print(f"Shape of the original dataset: {df_original.shape} (rows, columns)")

ma_conditions = {
    'MA1': ('c4', -0.9, 2),
    'MA2': ('c5', -0.4, 1),
    'MA3': ('c6', -1, 1),
}

ma_indices = {}
ma_label_counts = {}
for name, (feature, low, high) in ma_conditions.items():
    indices, counts = get_ma_indices(df_original, feature, low, high)
    ma_indices[name] = indices
    ma_label_counts[name] = counts
    print(f"{name} Condition: Feature '{feature}' in [{low}, {high}]")
    print(f"  samples = {len(indices)}")
    print(f"  label counts =\n{counts}\n")

Shape of the original dataset: (11183, 7) (rows, columns)
MA1 Condition: Feature 'c4' in [-0.9, 2]
  samples = 10951
  label counts =
label
1    10747
2      204
Name: count, dtype: int64

MA2 Condition: Feature 'c5' in [-0.4, 1]
  samples = 10098
  label counts =
label
1    10022
2       76
Name: count, dtype: int64

MA3 Condition: Feature 'c6' in [-1, 1]
  samples = 8263
  label counts =
label
1    8209
2      54
Name: count, dtype: int64



#### ⚪ 3. Create X_tem lists for MA1/MA2/MA3

In [4]:
X_tem_ma1 = ma_indices['MA1'].tolist()
X_tem_ma2 = ma_indices['MA2'].tolist()
X_tem_ma3 = ma_indices['MA3'].tolist()

print('MA1 size:', len(X_tem_ma1))
print('MA2 size:', len(X_tem_ma2))
print('MA3 size:', len(X_tem_ma3))

MA1 size: 10951
MA2 size: 10098
MA3 size: 8263


In [5]:
# Three-way neighborhood characteristic region-based outlier detection (3WIROD) algorithm
# Please refer to the following papers:
# Zhang Xianyong, Yuan Zhong, and Miao Duoqian.Outlier Detection Using Three-Way
# Neighborhood Characteristic Regions and Corresponding Fusion Measurement[J].TKDE,2023.
# Uploaded by Yuan Zhong on August 29, 2023. E-mail:yuanzhong2799@foxmail.com.

def WNCROD(data, X_tem, lammda):
    # input:
    # data is data matrix without decisions, where rows for samples and columns for attributes.
    # Numerical attributes should be normalized into [0,1].
    # Nominal attributes be replaced by different integer values.
    # X_tem denotes the selected condition subdata.
    # lammda is a given parameter for the radius adjustment.
    # output
    # Multiple neighborhood outlier factor (MNOF)
    n, m = data.shape
    X = np.zeros(n)
    X[X_tem] = 1

    D1 = m / 3
    D2 = m / 2
    D3 = 0.9 * m

    delta = np.zeros(m)
    ID = np.all(data <= 1, axis=0)
    delta[ID] = np.std(data[:, ID], axis=0) / lammda

    Lower = np.zeros((m, n))

    for col in range(m):
        RM_tem = cdist(data[:, [col]], data[:, [col]], metric='cityblock') <= delta[col]
        Lower_temp = np.min(np.maximum(1 - RM_tem, np.tile(X, (n, 1))), axis=1)
        Lower[col, :Lower_temp.shape[0]] = Lower_temp

    IB = np.tile(X, (m, 1)) - Lower
    NEB = np.min(IB, axis=0)
    NPB = IB - np.tile(NEB, (m, 1))

    n_X = int(sum(X))

    weight = np.zeros((n_X, m))

    for col in range(m):
        RM_tem = cdist(data[:, [col]], data[:, [col]], metric='cityblock') <= delta[col]
        weight_x = []

        for i in range(n_X):
            temp1 = RM_tem[X_tem[i], :]
            weight_temp = 1 - (np.sqrt((np.sum(np.minimum(temp1, X))) / n_X))
            weight_x.append(weight_temp)

        weight[:len(weight_x), col] = weight_x

    D_tem = np.zeros((n, n))

    for col in range(m):
        RM_tem = cdist(data[:, [col]], data[:, [col]], metric='cityblock') <= delta[col]
        D_tem += RM_tem

    NOM = m - D_tem
    X_OM = NOM[np.ix_(X_tem, X_tem)]

    NEB_num = np.zeros((n_X, m))
    Lower_num = np.zeros((n_X, m))
    NPB_num = np.zeros((n_X, m))

    for col in range(m):
        temp2 = Lower[col, :]
        temp3 = NPB[col, :]

        for i in range(n_X):
            temp1 = X_OM[i, :]
            NEB_num[i, col] = np.sum(np.minimum(NEB[X_tem], temp1 <= D1))
            NPB_num[i, col] = np.sum(np.minimum(temp3[X_tem], temp1 >= D2))
            Lower_num[i, col] = np.sum(np.minimum(temp2[X_tem], temp1 >= D3))

    MNOF = np.mean(((NEB_num + Lower_num + NPB_num) / n_X) * weight, axis=1)
    return MNOF


#### ⚪ 4. Run WNCROD for MA1, MA2, and MA3

In [7]:
# 1) Load data and normalize once
mat = loadmat('/media/mine/Mine/3.Main-Ubuntu-Envs/2.Thesis-Env/3.Datasets/mammography.mat')
full_data = mat['trandata']            # shape = (n_samples, 7)

X = full_data[:, :6]
y = full_data[:, 6].astype(int)

scaler = MinMaxScaler()
X[:, 1:] = scaler.fit_transform(X[:, 1:])

print('Loaded and normalized data once for all MA subsets.')

def evaluate_wncrod(data, labels, X_tem, lam=0.5):
    out_scores = WNCROD(data, X_tem, lammda=lam)
    return roc_auc_score(labels[X_tem], out_scores), average_precision_score(labels[X_tem], out_scores)

print('Running WNCROD for MA1, MA2, MA3 with λ = 0.5')
for name, X_tem in [('MA1', X_tem_ma1), ('MA2', X_tem_ma2), ('MA3', X_tem_ma3)]:
    roc_auc, ap = evaluate_wncrod(X, y, X_tem, lam=0.5)
    print(f"{name}: ROC-AUC = {roc_auc:.4f}, AP = {ap:.4f}")


Loaded and normalized data once for all MA subsets.
Running WNCROD for MA1, MA2, MA3 with λ = 0.5
MA1: ROC-AUC = 0.8931, AP = 0.3550
MA2: ROC-AUC = 0.7745, AP = 0.0626
MA3: ROC-AUC = 0.7748, AP = 0.0827


#### ⚪ 5. Run WNCROD only on MA1 for λ = 0.1 to 2.0

In [8]:
lambda_list_ma1 = [0.1, 0.2, 0.3, 0.4, 0.5,
                   0.6, 0.7, 0.8, 0.9, 1.0,
                   1.1, 1.2, 1.3, 1.4, 1.5,
                   1.6, 1.7, 1.8, 1.9, 2.0]

ma1_results = []
for lam in lambda_list_ma1:
    roc_auc, ap = evaluate_wncrod(X, y, X_tem_ma1, lam=lam)
    ma1_results.append({'lambda': lam, 'ROC-AUC': roc_auc, 'AP': ap})

print('MA1 results for λ = 0.1 تا 2.0:')
for r in ma1_results:
    print(f"λ = {r['lambda']:.1f} | ROC-AUC = {r['ROC-AUC']:.4f} | AP = {r['AP']:.4f}")

MA1 results for λ = 0.1 تا 2.0:
λ = 0.1 | ROC-AUC = 0.3409 | AP = 0.0168
λ = 0.2 | ROC-AUC = 0.4595 | AP = 0.0509
λ = 0.3 | ROC-AUC = 0.6006 | AP = 0.0774
λ = 0.4 | ROC-AUC = 0.7606 | AP = 0.1505
λ = 0.5 | ROC-AUC = 0.8931 | AP = 0.3550
λ = 0.6 | ROC-AUC = 0.8874 | AP = 0.2114
λ = 0.7 | ROC-AUC = 0.8610 | AP = 0.1315
λ = 0.8 | ROC-AUC = 0.8548 | AP = 0.1018
λ = 0.9 | ROC-AUC = 0.8506 | AP = 0.0905
λ = 1.0 | ROC-AUC = 0.8448 | AP = 0.0809
λ = 1.1 | ROC-AUC = 0.8277 | AP = 0.0748
λ = 1.2 | ROC-AUC = 0.8196 | AP = 0.0685
λ = 1.3 | ROC-AUC = 0.8154 | AP = 0.0640
λ = 1.4 | ROC-AUC = 0.8082 | AP = 0.0601
λ = 1.5 | ROC-AUC = 0.8032 | AP = 0.0567
λ = 1.6 | ROC-AUC = 0.7934 | AP = 0.0530
λ = 1.7 | ROC-AUC = 0.7858 | AP = 0.0500
λ = 1.8 | ROC-AUC = 0.7787 | AP = 0.0474
λ = 1.9 | ROC-AUC = 0.7761 | AP = 0.0457
λ = 2.0 | ROC-AUC = 0.7727 | AP = 0.0442


#### ⚪ 6. Run WNCROD for All MA for λ = 0.1 to 2.0

In [ ]:
lambda_list = [0.1, 0.2, 0.3, 0.4, 0.5,
               0.6, 0.7, 0.8, 0.9, 1.0,
               1.1, 1.2, 1.3, 1.4, 1.5,
               1.6, 1.7, 1.8, 1.9, 2.0]

all_results = {name: [] for name in ['MA1', 'MA2', 'MA3']}
for lam in lambda_list:
    for name, X_tem in [('MA1', X_tem_ma1), ('MA2', X_tem_ma2), ('MA3', X_tem_ma3)]:
        roc_auc, ap = evaluate_wncrod(X, y, X_tem, lam=lam)
        all_results[name].append({'lambda': lam, 'ROC-AUC': roc_auc, 'AP': ap})

for name in ['MA1', 'MA2', 'MA3']:
    print(f'\n{name} results:')
    for r in all_results[name]:
        print(f"λ = {r['lambda']:.1f} | ROC-AUC = {r['ROC-AUC']:.4f} | AP = {r['AP']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

lambda_list = [0.1, 0.2, 0.3, 0.4, 0.5,
               0.6, 0.7, 0.8, 0.9, 1.0,
               1.1, 1.2, 1.3, 1.4, 1.5,
               1.6, 1.7, 1.8, 1.9, 2.0]

roc_auc_ma1 = [r['ROC-AUC'] for r in all_results['MA1']]

plt.figure(figsize=(8, 5))
plt.plot(lambda_list, roc_auc_ma1, marker='o', linestyle='-',
         color='C0', label='ROC-AUC')
plt.title('ROC-AUC vs λ (MA1)')
plt.xlabel('λ')
plt.ylabel('ROC-AUC')
plt.xticks(lambda_list, rotation=45)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### 🔴 Section 2: Implement Feature-Wise for MA's

In [14]:
def WNCROD_featurewise(data, X_tem, lambdas):
    """Feature-wise 3WNCROD: each feature gets its own λ value.

    This version keeps each feature's neighborhood radius independent
    by computing delta[col] using sigma(feature) / lambda_feature.
    """
    n, m = data.shape
    lambdas = np.asarray(lambdas, dtype=float).reshape(-1)
    if lambdas.shape[0] != m:
        raise ValueError(f'Expected lambdas shape ({m},), got {lambdas.shape}')

    X = np.zeros(n)
    X[X_tem] = 1

    D1 = m / 3
    D2 = m / 2
    D3 = 0.9 * m

    delta = np.zeros(m)
    ID = np.all(data <= 1, axis=0)
    delta[ID] = np.std(data[:, ID], axis=0) / lambdas[ID]

    Lower = np.zeros((m, n))
    for col in range(m):
        RM_tem = cdist(data[:, [col]], data[:, [col]], metric='cityblock') <= delta[col]
        Lower_temp = np.min(np.maximum(1 - RM_tem, np.tile(X, (n, 1))), axis=1)
        Lower[col, :Lower_temp.shape[0]] = Lower_temp

    IB = np.tile(X, (m, 1)) - Lower
    NEB = np.min(IB, axis=0)
    NPB = IB - np.tile(NEB, (m, 1))

    n_X = int(sum(X))
    weight = np.zeros((n_X, m))
    for col in range(m):
        RM_tem = cdist(data[:, [col]], data[:, [col]], metric='cityblock') <= delta[col]
        weight_x = []
        for i in range(n_X):
            temp1 = RM_tem[X_tem[i], :]
            weight_temp = 1 - (np.sqrt((np.sum(np.minimum(temp1, X))) / n_X))
            weight_x.append(weight_temp)
        weight[:len(weight_x), col] = weight_x

    D_tem = np.zeros((n, n))
    for col in range(m):
        RM_tem = cdist(data[:, [col]], data[:, [col]], metric='cityblock') <= delta[col]
        D_tem += RM_tem

    NOM = m - D_tem
    X_OM = NOM[np.ix_(X_tem, X_tem)]

    NEB_num = np.zeros((n_X, m))
    Lower_num = np.zeros((n_X, m))
    NPB_num = np.zeros((n_X, m))

    for col in range(m):
        temp2 = Lower[col, :]
        temp3 = NPB[col, :]
        for i in range(n_X):
            temp1 = X_OM[i, :]
            NEB_num[i, col] = np.sum(np.minimum(NEB[X_tem], temp1 <= D1))
            NPB_num[i, col] = np.sum(np.minimum(temp3[X_tem], temp1 >= D2))
            Lower_num[i, col] = np.sum(np.minimum(temp2[X_tem], temp1 >= D3))

    MNOF = np.mean(((NEB_num + Lower_num + NPB_num) / n_X) * weight, axis=1)
    return MNOF

In [15]:
def evaluate_new_wncrod(data, labels, X_tem, lambdas):
    out_scores = WNCROD_featurewise(data, X_tem, lambdas)
    return {
        'roc_auc': roc_auc_score(labels[X_tem], out_scores),
        'ap': average_precision_score(labels[X_tem], out_scores)
    }

In [16]:
dataset_configs = {
    'MA1': {
        'X_tem': X_tem_ma1,
        'AP': [0.3, 0.05, 0.05, 0.5, 0.7, 0.5],
        'ROC-AUC': [0.5, 0.1, 0.1, 0.6, 0.7, 0.5],
    },
    'MA2': {
        'X_tem': X_tem_ma2,
        'AP': [0.4, 2.0, 0.1, 0.4, 1.0, 0.4],
        'ROC-AUC': [1.3, 0.1, 0.1, 0.6, 0.8, 0.4],
    },
    'MA3': {
        'X_tem': X_tem_ma3,
        'AP': [0.5, 2.0, 0.9, 0.4, 0.5, 2.0],
        'ROC-AUC': [0.5, 1.1, 0.8, 0.4, 0.5, 0.6],
    },
}

for dataset_name, config in dataset_configs.items():
    print(f'\n=== {dataset_name} ===')
    for metric_name in ['AP', 'ROC-AUC']:
        metrics = evaluate_new_wncrod(X, y, config['X_tem'], config[metric_name])
        print(f'{metric_name} | lambda = {config[metric_name]}')
        print(f'  ROC-AUC = {metrics["roc_auc"]:.4f} | AP = {metrics["ap"]:.4f}')


=== MA1 ===
AP | lambda = [0.3, 0.05, 0.05, 0.5, 0.7, 0.5]
  ROC-AUC = 0.8944 | AP = 0.5648
ROC-AUC | lambda = [0.5, 0.1, 0.1, 0.6, 0.7, 0.5]
  ROC-AUC = 0.9191 | AP = 0.5074

=== MA2 ===
AP | lambda = [0.4, 2.0, 0.1, 0.4, 1.0, 0.4]
  ROC-AUC = 0.6565 | AP = 0.1220
ROC-AUC | lambda = [1.3, 0.1, 0.1, 0.6, 0.8, 0.4]
  ROC-AUC = 0.8322 | AP = 0.0939

=== MA3 ===
AP | lambda = [0.5, 2.0, 0.9, 0.4, 0.5, 2.0]
  ROC-AUC = 0.7758 | AP = 0.3184
ROC-AUC | lambda = [0.5, 1.1, 0.8, 0.4, 0.5, 0.6]
  ROC-AUC = 0.7988 | AP = 0.2701


In [ ]:
# If you want to reuse the same evaluation block later with a different lambda vector,
# call evaluate_new_wncrod(X, y, X_tem_ma1, new_lambdas) or similar.

In [ ]:
# This cell is intentionally left empty for future experiments.